In [ ]:
import json
from pathlib import Path
import requests
import xarray as xr
import numpy as np
import datetime
import pandas as pd
from erddapy import ERDDAP

### Authentication

You can add your auth in a file `secrets.json` in the home directory of this repo

In [ ]:
oceanops_auth = {'X-OceanOPS-Metadata-ID': 'nnnn',
 'X-OceanOPS-Metadata-Token': '16-char-secret-token'}

if Path('../secrets.json').exists():
    with open('../secrets.json') as f:
        oceanops_auth = json.load(f)
else:
    print("enter your oceanOPS authentication in the dictionary `oceanops_auth` in this cell")

# Request a WIGOS ID

Here we request a GOOS passport for a glider using an exising GTS ID. Update the `json_data` dictionary with the relevant program, model, start & end dates, and GTS ID of your glider

In [ ]:
url = 'https://www.ocean-ops.org/api/data/platforms/getid'

headers = {
    'Content-Type': 'application/json' ,
    'accept': 'application/json' ,
    **oceanops_auth
}

json_data =   [{
    "program": "VOTO: Ocean Knowledge glider program",
    "model": "SeaExplorer X2",
    "startDate": "2026-04-15T09:21:56",
    "endDate": "2026-04-21T06:26:43",
    "gtsId": "6801506"
          
  }]
response = requests.post(url, headers=headers, json=json_data)
print("Status", response.status_code)
resp = response.json()[0]
print(resp)

### An example sucessful response

```
[{'success': True,
  'message': '',
  'wigosRef': '0-22000-22-6801506',
  'ref': '6801506_022',
  'gtsId': '6801506',
  'input': {'gtsId': '6801506',
   'program': 'VOTO: Ocean Knowledge glider program',
   'model': 'SeaExplorer X2',
   'batchStatus': None,
   'longitude': None,
   'latitude': None,
   'serialNo': None,
   'name': None,
   'internalId': None,
   'startDate': '2026-04-15T09:21:56',
   'endDate': '2026-04-21T06:26:43'},
  'batchRequestRef': '2026-04-21T10:12:00Z-SeaExplorer X2',
  'model': 'SeaExplorer X2'}]
```


### An unsucessful response (note it's still a 200)

```
[{'success': False,
  'message': 'Provided GTS identifier is already allocated on the date range specified',
  'wigosRef': None,
  'ref': None,
  'gtsId': None,
  'input': {'gtsId': '6801506',
   'program': 'VOTO: Ocean Knowledge glider program',
   'model': 'SeaExplorer X2',
   'batchStatus': None,
   'longitude': None,
   'latitude': None,
   'serialNo': None,
   'name': None,
   'internalId': None,
   'startDate': '2026-04-15T09:21:56',
   'endDate': '2026-04-21T06:26:43'},
  'batchRequestRef': None,
  'model': None}]
```

# Request WIGOS Passports for all missions

Here we take a few steps:

1. Make a list of available OG1 format glider missions on the VOTO ERDDAP
2. Loop through those missions extracting WMO ID, start time & end time
3. Use this WMO ID and times to request a WIGOS passport ID from the ocean-ops endpoint
4. Store this WMO ID for usage in making tables

In [ ]:
def init_erddap(protocol="tabledap"):
    e = ERDDAP(
        server="https://erddap.observations.voiceoftheocean.org/erddap",
        protocol=protocol,
    )
    return e
e = init_erddap()


In [ ]:
with open('wigos.json', 'r') as f:
    dataset_passport_ids = json.load(f)
    
url = 'https://www.ocean-ops.org/api/data/platforms/getid'
headers = {
    'Content-Type': 'application/json' ,
    'accept': 'application/json' ,
    **oceanops_auth
}
def request_all_passports_voto_gliders(dataset_passport_ids):
    e.response = "csv"
    e.dataset_id = "allDatasets"
    df_datasets = e.to_pandas()
    df_datasets.set_index("datasetID", inplace=True)
    df_datasets = df_datasets[df_datasets.index.str.contains("OG_nrt")]
    available_datasets = df_datasets.index.str[7:].values
    for dataset_id in available_datasets:
        if dataset_id in dataset_passport_ids.keys():
            continue
        e.dataset_id = f"OG_nrt_{dataset_id}"
        ds = e.to_xarray()
        attrs = ds.attrs
        wmo_id  = str(int(attrs['wmo_id']))
        min_time = df_datasets.loc[f"OG_nrt_{dataset_id}", 'minTime (UTC)'][:19]
        max_time = df_datasets.loc[f"OG_nrt_{dataset_id}", 'maxTime (UTC)'][:19]
        
        json_data =   [{
            "program": "VOTO: Ocean Knowledge glider program",
            "model": "SeaExplorer X2",
            "startDate": min_time,
            "endDate": max_time,
            "gtsId": wmo_id
                  
          }]
        response = requests.post(url, headers=headers, json=json_data)
        resp = response.json()[0]
        print(dataset_id, resp['message'])
    
        if resp['success']:
            dataset_passport_ids[dataset_id] = resp['wigosRef']
            
            print("SUCCESS!", resp)
dataset_passport_ids = request_all_passports_voto_gliders(dataset_passport_ids)

with open('wigos.json', 'w') as f:
    json.dump(dataset_passport_ids, f)

In [ ]:
dataset_passport_ids = {
'SEA056_M103': '0-22000-16-6801675',
 'SEA067_M86': '0-22000-30-6801507',
 'SEA069_M60': '0-22000-22-6801506',
 'SEA068_M47': '0-22000-9-6801602',
 'SHW002_M22': '0-22000-1-8901183',
 'SEA076_M25': '0-22000-21-6801665',
 'SEA069_M23': '0-22000-23-6801506',
 'SEA055_M98': '0-22000-31-6801674',
 'SEA057_M75': '0-22000-2-2801950',
 'SEA045_M102': '0-22000-28-6801673',
 'SEA063_M69': '0-22000-30-6801707',
 'SEA044_M109': '0-22000-29-6801672',
 'SEA067_M81': '0-22000-31-6801507',
 'SEA079_M18': '0-22000-16-8901001',
 'SEA066_M72': '0-22000-14-6801505',
 'SHW001_M33': '0-22000-2-8901081',
 'SEA077_M12': '0-22000-23-8901050',
 'SEA056_M100': '0-22000-17-6801675',
 'SEA078_M15': '0-22000-14-8901000',
}

# Generate tables for ERDDAP that oceanOPS will harvest

This is an example pulling from the VOTO ERDDAP. ymmv!

Using the CSV column names from https://www.ocean-ops.org/passports/#/submit

We work through the WIGOS Passport ids that we have, extract metadata from the ERDDAP dataset of each mission, and compile these metadata into two tables, following the example from GOOS

In [ ]:
passport_dicts = []
sensor_dicts = []
for dataset_id, passport_id in dataset_passport_ids.items():
    e.dataset_id = f"OG_nrt_{dataset_id}"
    ds = e.to_xarray()

    attrs = ds.attrs
    
    platform_serial = dataset_id.split('_M')[0]
    passport_dict = {'PASSPORT IDENTIFIER': passport_id,
                     'PROGRAM': 'VOTO: Ocean Knowledge glider program',
                     'NETWORK': '<unknown>',
                     'PLATFORM MODEL': 'SeaExplorer X2',
                     'PLATFORM SERIAL NUMBER': platform_serial,
                     'DEPLOYMENT DATE': str(ds.time.values[~np.isnan(ds.time)][0])[:19] + 'Z',
                     'DEPLOYMENT LATITUDE': ds.latitude.values[~np.isnan(ds.latitude)][0],
                     'DEPLOYMENT LONGITUDE': ds.longitude.values[~np.isnan(ds.longitude)][0],
                     'DEPLOYMENT SHIP': 'Seeker', # tbd
                     'GTS ID': int(attrs['wmo_id']),
                     'TELECOM TYPE': 'iridium',                    
    }
    
    # conditional entries
    conditional_dict = {
        'PRINCIPAL INVESTIGATOR': 'Louise Biddle',
        'AGENCIES': 'VOTO',
        'AGENCY_ROLES': 'operating agency',
        'CONTRIBUTORS': attrs['contributor_name'],
        'CONTRIBUTOR_ROLE': attrs['contributor_role']
        
    }
    end_time = ds.time.values[~np.isnan(ds.time)][-1]
    
    if end_time < np.datetime64(datetime.datetime.now() - datetime.timedelta(days=3)):
        passport_dict['END DATE'] = str(ds.time.values[~np.isnan(ds.time)][-1])[:19] + 'Z'
        passport_dict['ENDING CAUSE'] = 'recovered'
    passport_dict = passport_dict | conditional_dict
    passport_dicts.append(passport_dict)  

    from string import ascii_uppercase as auc
    i = 0
    for var_name in ds.variables:
        variable = ds[var_name]
        attrs = variable.attrs
        if 'sensor' not in attrs.keys():
            continue
        if attrs['sensor'][:6] != 'SENSOR':
            continue
        sensor_dict = ds[attrs['sensor']].attrs
        if 'standard_name' not in attrs.keys():
            continue
        sensor_dict = {
            'PASSPORT IDENTIFIER': passport_id,
            'SHORTCODE': auc[i],
            'SENSOR MODEL': sensor_dict['sensor_model'],
            'VARIABLE': attrs['standard_name'],
            'SERIAL NUMBER': sensor_dict['sensor_serial_number'],
        }
        i += 1
        sensor_dicts.append(sensor_dict)
        

In [ ]:
df_passport = pd.DataFrame(passport_dicts)
df_passport.to_csv('passport.csv', sep=';', index=False)
df_passport

In [ ]:
df_sensor = pd.DataFrame(sensor_dicts)
df_sensor.to_csv('sensor.csv', sep=';', index=False)

df_sensor